# 1.导入依赖与环境设置

## 导入依赖

In [1]:
# 1. Python 标准库
import os
import time
import copy
import heapq
import random
import glob
import warnings
from datetime import datetime
from collections import Counter

# 2. 数据处理与可视化工具
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm import tqdm          # 进度条
from IPython.display import clear_output  # 用于动态刷新屏幕

# 3. 机器学习工具 
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

# 4. PyTorch 深度学习框架
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset, Dataset
from torchvision import datasets, models, transforms

## 固定随机种子

In [2]:
# 设置随机种子
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True 
    torch.backends.cudnn.benchmark = False

# 调用
seed_everything(42)

In [3]:
# 检查是否有GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 屏蔽警告
warnings.filterwarnings("ignore")

Using device: cpu


# 2.定义超参数与路径

## 相关文件路径

In [4]:
# === 配置区域 ===
DATA_DIR = './data'  # 数据根目录

# 1. 图像数据源配置
ALL_DATA_DIR = os.path.join(DATA_DIR, 'all_data') 

# 2. 元数据 CSV 路径 (用于 Group K-Fold 防止泄露)
CSV_PATH = os.path.join(DATA_DIR, 'HAM10000_metadata.csv')


# 动态生成保存目录 (按启动时间)
# 1. 获取当前时间，精确到分钟 (格式: 20260124_1705)
current_time = datetime.now().strftime("%Y%m%d_%H%M")

# 2. 拼接目录: ./model_save_group/20260124_1705
SAVE_DIR = os.path.join('./model_save', current_time)

# 3. 创建目录
os.makedirs(SAVE_DIR, exist_ok=True)


## 超参数

In [5]:
k_folds = 3               # 3折交叉验证
NUM_EPOCHS = 50           # 训练轮数
BATCH_SIZE = 64           # 批大小
PATIENCE = 10             # 耐心值
LEARNING_RATE = 1e-4      # 学习率
WEIGHT_DECAY = 1e-3       # 权重衰减

# 3.数据预处理与加载

## 重采样

In [6]:
# 辅助类，让同一个图片在作为训练集时应用强增强，作为验证集时只做Resize
class TransformSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

In [7]:
def get_sampler(dataset, mode='sqrt'):
    """
    计算加权采样器 (兼容 ImageFolder, Subset, TransformSubset)
    """
    # --- 1. 智能提取标签列表 (核心修改) ---
    def extract_targets(ds):
        # 情况A: 自定义 TransformSubset
        if hasattr(ds, 'subset'):
            return extract_targets(ds.subset)
        
        # 情况B: PyTorch Subset (最常见)
        if hasattr(ds, 'indices') and hasattr(ds, 'dataset'):
            # 通过索引去父数据集查标签
            parent_targets = ds.dataset.targets
            return [parent_targets[i] for i in ds.indices]
            
        # 情况C: 原始 ImageFolder
        if hasattr(ds, 'targets'):
            return ds.targets
            
        raise ValueError(f"无法提取标签，Dataset类型 {type(ds)} 不支持")

    # 获取当前 split 的所有标签
    targets = extract_targets(dataset)
    # ------------------------------------
    
    # 2. 统计每个类别的样本数
    # list 转 numpy 以便使用 bincount (如果 targets 不是 int 列表会报错，但在 ImageFolder 里通常是 int)
    targets = np.array(targets)
    # 注意：bincount 统计的是 0~max_label 的数量，确保 label 是连续的 0~6
    class_counts = np.bincount(targets)
    
    # 3. 计算类别权重
    if mode == 'inverse':
        # 避免除以 0 (虽然在这个数据集中不太可能，但工程上要防御)
        class_weights = 1.0 / (class_counts + 1e-6)
    elif mode == 'sqrt':
        class_weights = 1.0 / (np.sqrt(class_counts) + 1e-6)
    else:
        raise ValueError("mode must be 'inverse' or 'sqrt'")
        
    # 4. 为【每一个样本】分配权重
    sample_weights = np.array([class_weights[t] for t in targets])
    
    # 5. 转为 Tensor
    sample_weights = torch.from_numpy(sample_weights).float()
    
    # 6. 创建采样器
    sampler = WeightedRandomSampler(
        weights=sample_weights, 
        num_samples=len(sample_weights), 
        replacement=True
    )
    
    return sampler

## 数据增强

In [8]:
data_transforms = {
    'train': transforms.Compose([
        # 视野锁定
        transforms.Resize((224, 224)),
        
        # 镜像填充
        transforms.Pad(padding=50, padding_mode='reflect'),
        
        # 旋转
        transforms.RandomRotation(180),
        
        # 几何增强
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        
        # 切回原形
        transforms.CenterCrop((224, 224)),
        
        # 归一化
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

## 读取数据

In [9]:
# 加载数据 (transform=None 是对的，后面会动态挂载)
full_dataset = datasets.ImageFolder(ALL_DATA_DIR, transform=None)

# 3. 获取基础信息
all_labels = full_dataset.targets 
class_names = full_dataset.classes

print(f"样本分类: {class_names}")
print(f"总样本数: {len(full_dataset)}")

样本分类: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
总样本数: 9039


# 4.可视化检查

## 增强后图片可视化

In [10]:
# 1. 函数定义
def visualize_batch(dataloader):
    images, labels = next(iter(dataloader))
    plt.figure(figsize=(16, 8))
    for i in range(min(8, len(images))):
        ax = plt.subplot(2, 4, i + 1)
        # 反归一化以便显示
        img = images[i].numpy().transpose((1, 2, 0))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        plt.imshow(img)
        plt.title(f"Label: {class_names[labels[i]]}")
        plt.axis("off")
    plt.show()

# 2. 手动构建一个临时的 DataLoader 进行查看
# 目的：确保 data_transforms['train'] (旋转、镜像填充等) 工作正常

# 使用我们定义的 TransformSubset，把“全量数据”和“训练增强”绑在一起
# 注意：这里我们不需要切分，直接用 full_dataset 即可，因为我们只看一个 batch
viz_dataset = TransformSubset(full_dataset, transform=data_transforms['train'])

# 创建临时 loader (shuffle=True 保证能看到不同的图)
viz_loader = DataLoader(viz_dataset, batch_size=8, shuffle=True)

print("正在检查训练集图片质量 (应用了旋转与填充增强)...")
visualize_batch(viz_loader)

正在检查训练集图片质量 (应用了旋转与填充增强)...


UnidentifiedImageError: cannot identify image file <_io.BufferedReader name='./data\\all_data\\nv\\ISIC_0025806.jpg'>

## 重采样类别分布对比

In [ ]:
def visualize_resampling_effect(dataset, sampler, class_names=None):
    # --- 内部辅助函数：智能提取标签 ---
    def get_targets(ds):
        # 情况1: 如果是我们自定义的 TransformSubset，先剥一层
        if hasattr(ds, 'subset'):
            ds = ds.subset
            
        # 情况2: 如果是 PyTorch 的 Subset (有 indices 属性)
        if hasattr(ds, 'indices'):
            # 通过索引去父数据集查标签
            parent_targets = ds.dataset.targets
            return [parent_targets[i] for i in ds.indices]
            
        # 情况3: 如果是原生的 ImageFolder (有 targets 属性)
        if hasattr(ds, 'targets'):
            return ds.targets
            
        raise ValueError("无法提取标签：Dataset类型不支持")
    # ------------------------------------

    # 1. 获取原始数据集分布
    try:
        original_targets = get_targets(dataset)
    except Exception as e:
        print(f"Error: {e}")
        return

    original_counts = Counter(original_targets)
    
    # 2. 模拟重采样后的分布
    print("正在模拟重采样过程 (这可能需要几秒钟)...")
    # list(sampler) 会模拟生成一个 Epoch 的索引序列
    resampled_indices = list(sampler)
    
    # 注意：sampler 返回的是 dataset 中的索引
    # 我们需要根据这些索引再次去获取对应的标签
    # 这里有点绕：sampler 的索引是基于 dataset 的长度的
    
    # 这里的逻辑通过 "从 original_targets 里按索引取值" 最为稳妥
    # 因为 original_targets 已经是当前 dataset 的标签列表了
    resampled_targets = [original_targets[i] for i in resampled_indices]
    resampled_counts = Counter(resampled_targets)
    
    # 3. 准备绘图数据
    if class_names is None:
        # 尝试自动获取
        if hasattr(dataset, 'classes'):
            class_names = dataset.classes
        elif hasattr(dataset, 'dataset') and hasattr(dataset.dataset, 'classes'): # 处理Subset情况
            class_names = dataset.dataset.classes
        else:
            class_names = [str(i) for i in sorted(original_counts.keys())]
            
    labels = sorted(original_counts.keys())
    # 确保 class_names 长度足够
    if len(class_names) < len(labels):
         class_names = [str(i) for i in labels]
         
    labels_names = [class_names[i] for i in labels]
    
    original_values = [original_counts[i] for i in labels]
    resampled_values = [resampled_counts[i] for i in labels]
    
    # 4. 绘制对比饼状图
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    colors = plt.cm.Pastel1(np.linspace(0, 1, len(labels)))
    
    # 图1：原始分布
    axes[0].pie(original_values, labels=labels_names, autopct='%1.1f%%', 
                startangle=90, colors=colors, pctdistance=0.85)
    axes[0].set_title(f'Original Distribution\n(Total: {len(original_targets)})', fontsize=14, fontweight='bold')
    axes[0].add_artist(plt.Circle((0,0),0.70,fc='white'))

    # 图2：重采样后分布
    axes[1].pie(resampled_values, labels=labels_names, autopct='%1.1f%%', 
                startangle=90, colors=colors, pctdistance=0.85)
    axes[1].set_title(f'After Weighted Resampling\n(Total: {len(resampled_targets)})', fontsize=14, fontweight='bold')
    axes[1].add_artist(plt.Circle((0,0),0.70,fc='white'))

    plt.tight_layout()
    plt.show()
    
    print("-" * 60)
    print(f"{'Class':<20} | {'Original':<15} | {'Resampled':<15}")
    print("-" * 60)
    for i, name in enumerate(labels_names):
        print(f"{name:<20} | {original_values[i]:<15} | {resampled_values[i]:<15}")
    print("-" * 60)

In [ ]:
# ==========================================
# 调用示例：在全量数据上测试采样器效果
# ==========================================
print("\n正在检查重采样策略的效果...")

# 1. 临时计算全量数据的权重 (复用你之前的 get_sampler 逻辑核心)
# 注意：这里我们需要手动算一下，因为 get_sampler 可能还没定义或者不适用
targets = full_dataset.targets
class_counts = Counter(targets)

# 按照 'inverse' 模式计算权重
weights = []
for t in targets:
    # 避免除以0
    count = class_counts[t]
    weights.append(1.0 / count)
weights = torch.DoubleTensor(weights)

# 2. 创建一个临时的采样器
demo_sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(weights),
    replacement=True
)

# 3. 调用可视化函数
visualize_resampling_effect(full_dataset, demo_sampler)

# 5.定义模型

In [ ]:
# def get_model(num_classes):
#     # 加载预训练的ResNet50
#     model = models.resnet50(pretrained=True)
    
#     # 修改最后的全连接层
#     num_ftrs = model.fc.in_features
#     model.fc = nn.Linear(num_ftrs, num_classes)
    
#     return model

# vgg

In [ ]:
def get_model(model_name, num_classes):
    """
    根据名称动态获取模型
    model_name: 'resnet', 'vgg', 或 'vit'
    """
    if model_name == 'resnet':
        # 这里使用 resnet50 (你也可以改回 resnet101)
        model = models.resnet50(pretrained=True)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
        
    elif model_name == 'vgg':
        # 推荐使用带 Batch Normalization 的 vgg16，收敛更稳定
        model = models.vgg16_bn(pretrained=True)
        # VGG 的分类器最后一步在 classifier 模块的第 6 层
        num_ftrs = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(num_ftrs, num_classes)
        
    elif model_name == 'vit':
        # 使用基础版 ViT (Base 16)
        model = models.vit_b_16(pretrained=True)
        # ViT 的最后分类头叫作 heads.head
        num_ftrs = model.heads.head.in_features
        model.heads.head = nn.Linear(num_ftrs, num_classes)
        
    else:
        raise ValueError(f"未知的模型类型: {model_name}")
        
    return model

# 简单测试一下是否都能正常实例化
if __name__ == "__main__":
    dummy_classes = 7
    print("Testing ResNet...")
    m1 = get_model('resnet', dummy_classes)
    print("Testing VGG...")
    m2 = get_model('vgg', dummy_classes)
    print("Testing ViT...")
    m3 = get_model('vit', dummy_classes)
    print("模型加载函数修改成功！")

# 6.训练与验证函数

In [ ]:
# ==========================================
# 0. 【新增】Top-K 模型管理器 (中文版)
# ==========================================
class TopKCheckpointManager:
    def __init__(self, save_dir, fold_idx, k=3):
        self.save_dir = save_dir
        self.fold_idx = fold_idx
        self.k = k  # 保留前 K 个
        self.top_k_heap = [] 
        
    def update(self, model, f1_score, epoch):
        # 构造唯一文件名 (包含 Epoch 和 F1)
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"fold{self.fold_idx}_epoch{epoch}_f1_{f1_score:.4f}_{run_id}.pth"
        save_path = os.path.join(self.save_dir, filename)
        
        # --- 逻辑 A: 还没存满 K 个 ---
        if len(self.top_k_heap) < self.k:
            heapq.heappush(self.top_k_heap, (f1_score, save_path))
            torch.save(model.state_dict(), save_path)
            print(f"    [已存档 Top-{len(self.top_k_heap)}] F1分数: {f1_score:.4f}")
            
        # --- 逻辑 B: 存满了，但当前模型比堆里最差的那个要好 ---
        elif f1_score > self.top_k_heap[0][0]:
            # 1. 弹出堆顶（最差的那个），并删除物理文件
            worst_f1, worst_path = heapq.heappop(self.top_k_heap)
            if os.path.exists(worst_path):
                try:
                    os.remove(worst_path) # 删除旧文件，节省空间
                    print(f"    [已移除] 淘汰低分模型 F1: {worst_f1:.4f}")
                except Exception as e:
                    print(f"    警告: 删除文件失败 {e}")
            
            # 2. 压入新的，并保存文件
            heapq.heappush(self.top_k_heap, (f1_score, save_path))
            torch.save(model.state_dict(), save_path)
            print(f"    [已存档新 Top-{self.k}] F1分数: {f1_score:.4f}")
            
        # --- 逻辑 C: 比最差的还差 ---
        else:
            pass # 直接忽略，不保存

# ==========================================
# 1. 核心指标计算函数 (保持不变)
# ==========================================
def calculate_metrics(y_true, y_pred, y_probs):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    
    try:
        auc = roc_auc_score(y_true, y_probs, multi_class='ovr', average='macro')
    except:
        auc = 0.5
        
    cm = confusion_matrix(y_true, y_pred)
    sens = []
    spes = []
    num_classes = cm.shape[0]
    
    for i in range(num_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        
        sen = tp / (tp + fn) if (tp + fn) > 0 else 0
        spe = tn / (tn + fp) if (tn + fp) > 0 else 0
        sens.append(sen)
        spes.append(spe)
        
    macro_sen = np.mean(sens)
    macro_spe = np.mean(spes)
    
    return acc, f1, auc, macro_sen, macro_spe

# ==========================================
# 2. 适配 K-Fold 的训练函数 (中文输出 + 自动刷新)
# ==========================================
def train_model_cv(model, train_loader, val_loader, criterion, optimizer, 
                   fold_idx, num_epochs=25, patience=15, save_dir='./models'):
    
    # 确保保存目录存在
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # 初始化 Top-K 管理器
    checkpoint_manager = TopKCheckpointManager(save_dir, fold_idx, k=3)
    
    since = time.time()
    
    dataloaders = {'train': train_loader, 'val': val_loader}
    dataset_sizes = {'train': len(train_loader.dataset), 'val': len(val_loader.dataset)}
    
    global_best_f1 = 0.0
    epochs_no_improve = 0 
    early_stop = False
    
    # 用于记录简要历史，防止刷新后什么都看不见
    history_logs = []

    print(f"\n>>> 开始训练第 {fold_idx} 折 (Top-3 策略)")
    print(f">>> 早停耐心值: {patience}")

    for epoch in range(num_epochs):
        if early_stop:
            print(f"第 {fold_idx} 折: 触发早停机制，训练结束！")
            break
        
        # ================= [新增] 刷新逻辑 =================
        clear_output(wait=True) # 清除上一次输出
        
        # 重绘表头和历史记录（保留最近10条，避免太长）
        print(f"正在训练第 {fold_idx} 折 | 当前进度: {epoch+1}/{num_epochs}")
        print("-" * 75)
        print(f"{'轮次':<4} | {'阶段':<5} | {'Loss':<8} | {'F1':<8} | {'Acc':<8} | {'AUC':<8}")
        print("-" * 75)
        for log in history_logs[-10:]: # 只显示最近10条历史
            print(log)
        print("-" * 75)
        # ===================================================

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                phase_cn = "训练" # 中文转换
            else:
                model.eval()
                phase_cn = "验证"

            running_loss = 0.0
            all_preds = []
            all_labels = []
            all_probs = []

            # 进度条汉化
            loop = tqdm(dataloaders[phase], desc=f"[{phase_cn}阶段]", leave=False)
            
            for inputs, labels in loop:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    probs = torch.softmax(outputs, dim=1)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.detach().cpu().numpy())
                
                # 进度条显示 Loss
                loop.set_postfix(Current_Loss=f"{loss.item():.4f}")

            epoch_loss = running_loss / dataset_sizes[phase]
            
            # 计算指标
            epoch_acc, epoch_f1, epoch_auc, epoch_sen, epoch_spe = calculate_metrics(
                np.array(all_labels), np.array(all_preds), np.array(all_probs)
            )

            # 格式化当前轮次的日志字符串
            log_str = f"{epoch+1:<5} | {phase_cn:<5} | {epoch_loss:.4f}   | {epoch_f1:.4f}   | {epoch_acc:.4f}   | {epoch_auc:.4f}"
            history_logs.append(log_str) # 加入历史记录
            
            # 实时打印当前状态（因为上面clear了，这里要打印出来才看得到当前轮的结果）
            print(f"[{phase_cn}] Loss: {epoch_loss:.4f} | F1: {epoch_f1:.4f} | Acc: {epoch_acc:.4f}")

            # === 验证阶段逻辑 ===
            if phase == 'val':
                # 1. 尝试保存模型
                checkpoint_manager.update(model, epoch_f1, epoch+1)
                
                # 2. 早停逻辑
                if epoch_f1 > global_best_f1:
                    global_best_f1 = epoch_f1
                    epochs_no_improve = 0
                    print(f"  >>> 发现新高分 F1: {global_best_f1:.4f}")
                else:
                    epochs_no_improve += 1
                    print(f"  >>> 性能未提升 ({epochs_no_improve}/{patience})")
                    if epochs_no_improve >= patience:
                        early_stop = True

    time_elapsed = time.time() - since
    print(f'\n第 {fold_idx} 折训练完成，耗时 {time_elapsed // 60:.0f}分 {time_elapsed % 60:.0f}秒')
    
    return global_best_f1

# 开始运行

In [ ]:
# ==========================================
# 3. 启动 K-Fold 交叉验证 (最终正确版：StratifiedGroupKFold)
# 适用场景：样本量充足 (akiec > 200)，启用严格的病人级隔离
# ==========================================

# --- 1. 构建 Group 映射 (核心步骤) ---
print("  正在构建病灶分组 (Group Mapping)...")
df = pd.read_csv(CSV_PATH)

# 创建 image_id -> lesion_id 的字典
id_to_lesion = dict(zip(df['image_id'], df['lesion_id']))
    
groups = []
found_count = 0
    
for path, _ in full_dataset.samples:
    # 获取文件名 (不带后缀)
    filename = os.path.splitext(os.path.basename(path))[0]
    # 兼容性清洗：去掉可能存在的 train_ 前缀
    clean_filename = filename.replace('train_', '')
    
    # 尝试获取 lesion_id
    lesion_id = id_to_lesion.get(clean_filename)
        
    if lesion_id:
        groups.append(lesion_id)
        found_count += 1
    else:
        # 兜底策略：如果没有lesion_id，就认为是独立病人
        groups.append(clean_filename)
            
print(f"分组映射就绪! 匹配成功率: {found_count}/{len(full_dataset)} ({found_count/len(full_dataset):.2%})")

# --- 2. 初始化切分器 ---
# 使用 StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=k_folds, shuffle=True, random_state=42)
fold_results = []
X_dummy = np.zeros(len(all_labels))

# --- 3. K-Fold 循环 ---
# 注意：这里传入了 groups 参数
for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_dummy, all_labels, groups=groups)):
    
    print(f"\n>>> 第 {fold+1}/{k_folds} 折 准备中...")
    
    # 打印一下分布，确认 akiec 没有塌陷
    val_labels_fold = [all_labels[i] for i in val_idx]
    akiec_idx = class_names.index('akiec') if 'akiec' in class_names else 0
    akiec_count = val_labels_fold.count(akiec_idx)
    print(f"    验证集统计: akiec 样本数 = {akiec_count} (理论上应充足)")

    # A. 切分
    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)
    
    # B. 挂载增强 (TransformSubset 需要你在前面定义过)
    train_dataset = TransformSubset(train_subset, transform=data_transforms['train'])
    val_dataset = TransformSubset(val_subset, transform=data_transforms['val'])
    
    # C. 采样器 (WeightedRandomSampler)
    print("      正在计算类别权重 (平衡采样)...")
    # get_sampler 需要你在前面定义过
    train_sampler = get_sampler(train_dataset, mode='sqrt')
    
    # D. DataLoader
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, shuffle=False, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    
    # E. 模型初始化
    model = get_model(len(class_names)) 
    model = model.to(device)
    
    # F. 损失函数
    criterion = nn.CrossEntropyLoss()
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    
    # G. 开始训练 (调用刚才修改好的第一段代码)
    fold_best_f1 = train_model_cv(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        fold_idx=fold+1,
        num_epochs=NUM_EPOCHS,
        patience=PATIENCE,
        save_dir=SAVE_DIR
    )
    
    fold_results.append(fold_best_f1)
    print(f"     第 {fold+1} 折 结束. 本折最佳验证 F1: {fold_best_f1:.4f}")

# --- 4. 总结 ---
print(f"\n{'='*40}")
print(" 交叉验证最终报告 (分层分组策略)")
print(f"{'='*40}")
for i, f1 in enumerate(fold_results):
    print(f"第 {i+1} 折 F1: {f1:.4f}")

mean_f1 = np.mean(fold_results)
std_f1 = np.std(fold_results)
print(f"{'-'*40}")
print(f"平均 F1 分数: {mean_f1:.4f} ± {std_f1:.4f}")
print(f"{'='*40}")

# 7.Grad-CAM

In [ ]:
# ==========================================
# 1. Grad-CAM 核心类 (保持不变)
# ==========================================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # 注册钩子
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, input_tensor, class_idx=None):
        # 1. 前向传播
        self.model.zero_grad()
        output = self.model(input_tensor)
        
        if class_idx is None:
            class_idx = torch.argmax(output, dim=1)
            
        # 2. 反向传播
        score = output[0, class_idx]
        score.backward()
        
        # 3. 生成 CAM
        gradients = self.gradients
        activations = self.activations
        weights = torch.mean(gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * activations, dim=1, keepdim=True)
        cam = torch.relu(cam)
        
        # 归一化
        cam = cam - torch.min(cam)
        cam = cam / (torch.max(cam) + 1e-7)
        
        # 返回: 热力图, 预测类别索引, 置信度
        return cam.detach().cpu().numpy()[0, 0], class_idx.item(), torch.max(F.softmax(output, dim=1)).item()

# ==========================================
# 2. 【修改核心】集成批量绘图工具
# ==========================================
def plot_ensemble_gradcam(model_dir, img_paths, class_names):
    """
    Args:
        model_dir: 模型文件夹路径 (包含多个 .pth)
        img_paths: 图片路径列表
        class_names: 类别名称
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # --- A. 自动加载所有模型 ---
    search_path = os.path.join(model_dir, "*.pth")
    model_paths = glob.glob(search_path)
    model_paths.sort()

    print(f"正在加载 {len(model_paths)} 个模型用于集成 Grad-CAM...")
    
    # 准备 Grad-CAM 实例列表
    gradcam_instances = []
    
    for path in model_paths:
        # 实例化模型
        model = models.resnet50(pretrained=False)
        model.fc = torch.nn.Linear(model.fc.in_features, len(class_names))
        model.load_state_dict(torch.load(path, map_location=device))
        model = model.to(device)
        model.eval()
        
        # 为每个模型创建一个 Grad-CAM 解释器
        target_layer = model.layer4[-1]
        gradcam_instances.append(GradCAM(model, target_layer))

    # 预处理
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # --- B. 循环处理每一张图片 ---
    num_imgs = len(img_paths)
    # 动态调整画布大小
    fig, axes = plt.subplots(num_imgs, 3, figsize=(15, 5 * num_imgs))
    if num_imgs == 1: axes = [axes] 

    for i, img_path in enumerate(img_paths):
        img_pil = Image.open(img_path).convert('RGB')
        input_tensor = preprocess(img_pil).unsqueeze(0).to(device)
        
        # 获取真实标签
        true_label = img_path.split(os.sep)[-2] # 兼容 Windows/Linux 路径分隔符
        
        # --- C. 集成计算 (Ensemble Calculation) ---
        accumulated_cam = np.zeros((7, 7), dtype=np.float32) # ResNet50 特征图大小
        votes = []
        confs = []
        
        # 让每个模型都看一遍，累加热力图
        for gcam in gradcam_instances:
            # 获取单模型的热力图 (原始尺寸 7x7)
            # 注意：这里我们让 Grad-CAM 解释模型预测出的那个类，或者你可以强制解释 true_label
            mask, pred_idx, conf = gcam(input_tensor)
            
            # 累加
            accumulated_cam += mask
            votes.append(pred_idx)
            confs.append(conf)
            
        # 取平均热力图
        avg_cam = accumulated_cam / len(gradcam_instances)
        
        # 集成预测结果 (Soft Voting 简化版，这里用众数演示，或者也可以用 Softmax 平均)
        # 为了简单展示，我们这里取“投票最多”的类别作为集成预测结果
        final_pred_idx = max(set(votes), key=votes.count)
        final_pred_label = class_names[final_pred_idx]
        avg_conf_score = sum(confs) / len(confs)

        # --- D. 绘图 ---
        # 缩放平均热力图到原图大小
        mask_resized = cv2.resize(avg_cam, (img_pil.size[0], img_pil.size[1]))
        heatmap = cv2.applyColorMap(np.uint8(255 * mask_resized), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        overlay = np.uint8(np.array(img_pil) * 0.6 + heatmap * 0.4)
        
        # 1. 原图
        ax = axes[i][0] if num_imgs > 1 else axes[0]
        ax.imshow(img_pil)
        ax.set_title(f"True: {true_label}", fontsize=12)
        ax.axis('off')
        
        # 2. 集成热力图
        ax = axes[i][1] if num_imgs > 1 else axes[1]
        ax.imshow(heatmap)
        ax.set_title(f"Ensemble Heatmap\n(Avg of {len(gradcam_instances)} models)", fontsize=12)
        ax.axis('off')
        
        # 3. 叠加结果
        ax = axes[i][2] if num_imgs > 1 else axes[2]
        ax.imshow(overlay)
        color = 'green' if final_pred_label == true_label else 'red'
        ax.set_title(f"Ensemble Pred: {final_pred_label}\nConf: {avg_conf_score:.2%}", fontsize=12, color=color, fontweight='bold')
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()  

In [ ]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torchvision import datasets, transforms, models
# from torch.utils.data import DataLoader, random_split, Dataset
# import pandas as pd
# from tqdm import tqdm
# # 补充导入 sklearn 指标函数
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# # ==========================================
# # 1. 全局配置与超参数
# # ==========================================
# DATA_DIR = './data'  # 请替换为你真实的皮肤病图片文件夹路径
# BATCH_SIZE = 32         
# NUM_EPOCHS = 25          
# LEARNING_RATE = 1e-4
# NUM_CLASSES = 7         # 皮肤病分类数
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # ==========================================
# # 2. 数据准备与强数据增强 (分离训练集与验证集的 Transform)
# # ==========================================
# # 训练集：加入强数据增强（旋转、翻转、颜色抖动）来防止 VGG 死记硬背，逼出 ResNet 的泛化能力
# train_transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.RandomHorizontalFlip(p=0.5),      # 随机水平翻转
#     transforms.RandomVerticalFlip(p=0.5),        # 随机垂直翻转
#     transforms.RandomRotation(degrees=30),       # 随机旋转 ±30度
#     transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), # 颜色抖动
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# # 验证集：绝对不能增强，只做缩放和归一化
# val_transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# # 自定义 Dataset 包装类，用于给 random_split 切分出来的 Subset 分别应用不同的 Transform
# class TransformSubset(Dataset):
#     def __init__(self, subset, transform=None):
#         self.subset = subset
#         self.transform = transform

#     def __getitem__(self, index):
#         x, y = self.subset[index]
#         if self.transform:
#             x = self.transform(x)
#         return x, y

#     def __len__(self):
#         return len(self.subset)

# # 第一步：读取基础数据集（暂不加 transform，保持 PIL 图像格式）
# base_dataset = datasets.ImageFolder(root=DATA_DIR)

# # 第二步：按 8:2 随机划分
# train_size = int(0.8 * len(base_dataset))
# val_size = len(base_dataset) - train_size
# train_subset, val_subset = random_split(base_dataset, [train_size, val_size])

# # 第三步：给切分好的子集分别套上对应的 Transform
# train_dataset = TransformSubset(train_subset, transform=train_transform)
# val_dataset = TransformSubset(val_subset, transform=val_transform)

# # 第四步：构建 DataLoader
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# # ==========================================
# # 3. 模型构建字典 (去掉了 ViT)
# # ==========================================
# def get_models():
#     # ResNet50 (你毕设的绝对主角)
#     resnet = models.resnet50(pretrained=True)
#     resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
    
#     # VGG16 (陪跑基线)
#     vgg = models.vgg16(pretrained=True)
#     vgg.classifier[6] = nn.Linear(vgg.classifier[6].in_features, NUM_CLASSES)
    
#     return {"ResNet50": resnet, "VGG16": vgg}

# models_dict = get_models()

# # ==========================================
# # 4. 训练与评估逻辑
# # ==========================================
# results = {
#     "Model": [], 
#     "Best_Macro_F1": [], 
#     "Accuracy": [], 
#     "Macro_Precision": [], 
#     "Macro_Recall": []
# }

# for model_name, model in models_dict.items():
#     print(f"\n" + "="*50)
#     print(f"🚀 开始训练模型: {model_name}")
#     print("="*50)
    
#     model = model.to(DEVICE)
#     optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
#     criterion = nn.CrossEntropyLoss()
    
#     best_macro_f1 = 0.0
#     best_epoch_metrics = {} 
    
#     for epoch in range(NUM_EPOCHS):
#         # --- 训练阶段 ---
#         torch.set_grad_enabled(True)  
#         model.train()
        
#         for param in model.parameters():
#             param.requires_grad = True
            
#         running_loss = 0.0
#         for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"):
#             inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
#             optimizer.zero_grad()
#             outputs = model(inputs)
#             loss = criterion(outputs, labels)
#             loss.backward()
#             optimizer.step()
            
#             running_loss += loss.item() * inputs.size(0)
            
#         # --- 验证阶段 ---
#         model.eval()
        
#         all_preds = []
#         all_labels = []
        
#         with torch.no_grad():
#             for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
#                 inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
#                 outputs = model(inputs)
#                 _, preds = torch.max(outputs, 1)
                
#                 all_preds.extend(preds.cpu().numpy())
#                 all_labels.extend(labels.cpu().numpy())
                
#         # 计算各项常用评价指标
#         epoch_acc = accuracy_score(all_labels, all_preds)
#         epoch_macro_f1 = f1_score(all_labels, all_preds, average='macro')
#         epoch_precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
#         epoch_recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
        
#         print(f"[{model_name}] Epoch {epoch+1} | Acc: {epoch_acc:.4f} | Macro F1: {epoch_macro_f1:.4f} | Prec: {epoch_precision:.4f} | Rec: {epoch_recall:.4f}")
        
#         # 以 Macro F1 为标准保存最佳模型
#         if epoch_macro_f1 > best_macro_f1:
#             best_macro_f1 = epoch_macro_f1
            
#             best_epoch_metrics = {
#                 "acc": epoch_acc,
#                 "prec": epoch_precision,
#                 "rec": epoch_recall
#             }
#             torch.save(model.state_dict(), f"{model_name}_best.pth")
            
#     results["Model"].append(model_name)
#     results["Best_Macro_F1"].append(best_macro_f1)
#     results["Accuracy"].append(best_epoch_metrics.get("acc", 0.0))
#     results["Macro_Precision"].append(best_epoch_metrics.get("prec", 0.0))
#     results["Macro_Recall"].append(best_epoch_metrics.get("rec", 0.0))
    
#     print(f"{model_name} 训练完成，最佳验证集 Macro F1: {best_macro_f1:.4f}")

# # ==========================================
# # 5. 打印最终对比表格
# # ==========================================
# print("\n" + "="*60)
# print("各模型横向对比实验结果 (加入强数据增强后)")
# print("="*60)
# df_results = pd.DataFrame(results)
# print(df_results.to_string(index=False))